# Floriscan Phase B — blooming stage (BDFlower)

Train a **shared 3-class** stage model (`bud` / `partially_open` / `fully_open`) on top of your existing species EfficientNet-B0.

BDFlower's eight species are **not** your Floriscan ten. That is fine: we **pool** every Early/Mid/Full original into the three stage folders and throw away the species name. Flask still names the flower with `floriscan_species.pth`, then this head guesses openness.

- Use **originals only** (skip the paper's 5× Augmentation copies).
- Copy images onto **Colab local disk** (`/content/floriscan_stages`) so training is not Drive-bound.
- Transfer the backbone from Drive `exports/floriscan_species.pth` (or ImageNet if missing).
- Keep hibiscus and cherry blossom out of the Flask cascade until you check them.

**Must match Flask:** 384, ImageNet mean/std, stage order `bud`, `partially_open`, `fully_open`.

After training, copy `floriscan_stage.pth` to `D:\\Projects\\Floriscan\\backend\\models\\`.

If Colab GPU quota is blocked, use `scripts/train_stage.py` on **D:** instead (never C:).


In [ ]:
# Do not pin opencv 4.8 / numpy 1.26 — that breaks current Colab.
!pip -q install -U timm
!python -c "import numpy, torch, timm; print('numpy', numpy.__version__); print('torch', torch.__version__); print('timm', timm.__version__)"


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os
import random
import shutil
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

STAGES = ["bud", "partially_open", "fully_open"]
SPECIES_N = 10
STAGE_SUPPORTED = [
    "rose", "tulip", "lily", "sunflower",
    "carnation", "peony", "iris", "daffodil",
]
SKIP_FOR_NOW = ["hibiscus", "cherry_blossom"]

DRIVE_ROOT = Path("/content/drive/MyDrive/Floriscan")
EXPORT = DRIVE_ROOT / "exports"
DRIVE_ZIP = DRIVE_ROOT / "data" / "stages" / "bdflower.zip"
DRIVE_EXTRA = DRIVE_ROOT / "data" / "stages" / "extra"
LOCAL = Path("/content/floriscan_stage_work")
LOCAL_ZIP = LOCAL / "bdflower.zip"
EXTRACT = LOCAL / "extracted"
MERGED = Path("/content/floriscan_stages")

for name in STAGES:
    (MERGED / name).mkdir(parents=True, exist_ok=True)
    (DRIVE_EXTRA / name).mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
DRIVE_ZIP.parent.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 384
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MENDELEY_ZIP_URL = (
    "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/m8g2wynwyr-2.zip"
)
EARLY = {"early", "early stage", "earlystage", "early_stage"}
MID = {"mid", "mid stage", "midstage", "mid_stage"}
FULL = {"full", "full stage", "fullstage", "full_stage", "full growth", "fullgrowth"}


def _norm_part(name: str) -> str:
    return name.lower().replace("-", " ").replace("_", " ").strip()


def is_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTS and path.stat().st_size > 2048


def is_augmented(path: Path) -> bool:
    parts = [_norm_part(p) for p in path.parts]
    if any(p in {"augmentation", "augmented", "aug"} for p in parts):
        return True
    stem = path.stem.lower()
    return "_aug" in stem or "aug1" in stem or "aug2" in stem or "aug3" in stem


def map_stage(path: Path):
    if is_augmented(path):
        return None
    for part in path.parts:
        key = _norm_part(part)
        if key in EARLY:
            return "bud"
        if key in MID:
            return "partially_open"
        if key in FULL:
            return "fully_open"
    return None


def copy_unique(src: Path, dest_dir: Path, prefix: str) -> bool:
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"{prefix}_{src.name}"
    if dest.exists():
        return False
    shutil.copy2(src, dest)
    return True


def count_stage(name: str) -> int:
    folder = MERGED / name
    if not folder.is_dir():
        return 0
    return sum(1 for p in folder.iterdir() if is_image(p))


print("local merge", MERGED)
print("optional extra photos", DRIVE_EXTRA)
print("STAGE_SUPPORTED", STAGE_SUPPORTED)
print("held out", SKIP_FOR_NOW)


## 1. Get BDFlower onto Colab disk

Tries Drive zip first (so you only download once), then Mendeley's zip cache.
If both fail, download from https://data.mendeley.com/datasets/m8g2wynwyr/2 and put `bdflower.zip` in `MyDrive/Floriscan/data/stages/`.


In [ ]:
if DRIVE_ZIP.is_file() and DRIVE_ZIP.stat().st_size > 1_000_000:
    print("Copying zip from Drive (faster on reruns)")
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
elif LOCAL_ZIP.is_file() and LOCAL_ZIP.stat().st_size > 1_000_000:
    print("Using zip already on Colab disk")
else:
    print("Downloading BDFlower zip (one-time)")
    ok = os.system(f'wget -q --show-progress -O "{LOCAL_ZIP}" "{MENDELEY_ZIP_URL}"')
    if ok != 0 or not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size < 1_000_000:
        raise SystemExit(
            "Download failed. Get the zip from Mendeley in a browser and upload it to\n"
            f"  {DRIVE_ZIP}\nthen rerun this cell."
        )
    try:
        shutil.copy2(LOCAL_ZIP, DRIVE_ZIP)
        print("Cached zip on Drive for next time")
    except Exception as exc:
        print("Could not cache zip on Drive:", exc)

marker = EXTRACT / ".extracted"
if not marker.exists():
    EXTRACT.mkdir(parents=True, exist_ok=True)
    print("Extracting on local disk (not Drive)")
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(EXTRACT)
    marker.write_text("ok")
print("extracted", EXTRACT)


## 2. Pool Early / Mid / Full originals into 3 stage folders

Skip anything under `Augmentation` or `*_AugN.jpg`. Optional extras from Drive `data/stages/extra/<stage>/` (your rose/tulip photos) are copied in too.


In [ ]:
added = {name: 0 for name in STAGES}
skipped_aug = 0
skipped_other = 0
for path in EXTRACT.rglob("*"):
    if not is_image(path):
        continue
    if is_augmented(path):
        skipped_aug += 1
        continue
    stage = map_stage(path)
    if stage is None:
        skipped_other += 1
        continue
    added[stage] += int(copy_unique(path, MERGED / stage, "bdf"))
print("BDFlower originals", added)
print("skipped augmented", skipped_aug, "unmapped", skipped_other)

for name in STAGES:
    extra_dir = DRIVE_EXTRA / name
    n = 0
    if extra_dir.is_dir():
        for path in extra_dir.rglob("*"):
            if is_image(path):
                n += int(copy_unique(path, MERGED / name, "extra"))
    print("extra", name, "+", n)

print("Ready to train (local disk):")
for name in STAGES:
    print(f"  {name:16s} {count_stage(name)}")
thin = [name for name in STAGES if count_stage(name) < 80]
if thin:
    raise SystemExit("Need at least ~80 originals per stage. Thin: " + ", ".join(thin))


## 3. Split and loaders (read from `/content`, not Drive)


In [ ]:
class StageFolder(Dataset):
    def __init__(self, items, train=False):
        self.items = items
        self.train = train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, label = self.items[index]
        img = cv2.imread(str(path))
        if img is None:
            img = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        if self.train:
            if random.random() < 0.5:
                img = cv2.flip(img, 1)
            if random.random() < 0.35:
                angle = random.uniform(-12, 12)
                matrix = cv2.getRotationMatrix2D((IMAGE_SIZE / 2, IMAGE_SIZE / 2), angle, 1.0)
                img = cv2.warpAffine(img, matrix, (IMAGE_SIZE, IMAGE_SIZE), borderMode=cv2.BORDER_REFLECT)
        img = img.astype(np.float32) / 255.0
        img = (img - MEAN) / STD
        return torch.from_numpy(img).permute(2, 0, 1), label


train_items, val_items, test_items = [], [], []
print("per-class 70/15/15 split:")
for class_id, name in enumerate(STAGES):
    files = [p for p in (MERGED / name).iterdir() if is_image(p)]
    files.sort()
    random.shuffle(files)
    n = len(files)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_files = files[:n_train]
    val_files = files[n_train:n_train + n_val]
    test_files = files[n_train + n_val:]
    train_items.extend((path, class_id) for path in train_files)
    val_items.extend((path, class_id) for path in val_files)
    test_items.extend((path, class_id) for path in test_files)
    print(f"  {name:16s} n={n:4d}  train {len(train_files):4d}  val {len(val_files):4d}  test {len(test_files):4d}")

random.shuffle(train_items)
counts = np.zeros(len(STAGES), dtype=np.float64)
for _, label in train_items:
    counts[label] += 1
counts = np.maximum(counts, 1.0)
sample_w = [1.0 / counts[label] for _, label in train_items]
sampler = WeightedRandomSampler(sample_w, num_samples=len(train_items), replacement=True)

train_loader = DataLoader(StageFolder(train_items, train=True), batch_size=16, sampler=sampler, num_workers=2)
val_loader = DataLoader(StageFolder(val_items, train=False), batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(StageFolder(test_items, train=False), batch_size=16, shuffle=False, num_workers=2)
print("totals", len(train_items), len(val_items), len(test_items))


## 4. Transfer from species EfficientNet-B0 and train

Classifier is 3-way (not 10). Backbone weights come from your Phase A `.pth`.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)
if device.type != "cuda":
    print("CPU will be slow. Runtime → Change runtime type → GPU, or use scripts/train_stage.py on D:.")

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(STAGES))
species_path = EXPORT / "floriscan_species.pth"
if not species_path.is_file():
    species_path = Path("/content/drive/MyDrive/Floriscan/exports/floriscan_species.pth")
if species_path.is_file():
    try:
        donor = timm.create_model("efficientnet_b0", pretrained=False, num_classes=SPECIES_N)
        state = torch.load(species_path, map_location="cpu")
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        donor.load_state_dict(state)
        dst = model.state_dict()
        copied = 0
        for key, value in donor.state_dict().items():
            if key.startswith("classifier"):
                continue
            if key in dst and dst[key].shape == value.shape:
                dst[key] = value
                copied += 1
        model.load_state_dict(dst)
        print("Transferred backbone from", species_path, "tensors", copied)
    except Exception as exc:
        print("Could not transfer species weights, using ImageNet backbone:", exc)
else:
    print("No floriscan_species.pth on Drive — training 3-class head on ImageNet EfficientNet-B0")

model.to(device)
class_w = torch.tensor((counts.max() / counts).astype(np.float32), device=device)
print("class weights", {STAGES[i]: round(float(class_w[i]), 2) for i in range(len(STAGES))})
criterion = nn.CrossEntropyLoss(weight=class_w)

for name, param in model.named_parameters():
    param.requires_grad = name.startswith("classifier")
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


best_val = 0.0
best_path = EXPORT / "floriscan_stage.pth"
for epoch in range(6):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"head {epoch+1:02d}  train {train_acc:.3f}  val {val_acc:.3f}  val_loss {val_loss:.4f}")
    if val_acc >= best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)

for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
for epoch in range(8):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"ft   {epoch+1:02d}  train {train_acc:.3f}  val {val_acc:.3f}  val_loss {val_loss:.4f}")
    if val_acc >= best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)

print("best val", best_val, "saved", best_path)


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        y_true.extend(labels.tolist())
        y_pred.extend(model(images.to(device)).argmax(1).cpu().tolist())

label_ids = list(range(len(STAGES)))
print("test images per stage:")
for class_id, name in enumerate(STAGES):
    print(f"  {name:16s} {y_true.count(class_id)}")
print(
    classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=STAGES,
        digits=3,
        zero_division=0,
    )
)
matrix = confusion_matrix(y_true, y_pred, labels=label_ids)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(matrix, cmap="Greens")
ax.set_xticks(range(len(STAGES)), STAGES, rotation=20, ha="right")
ax.set_yticks(range(len(STAGES)), STAGES)
ax.set_title("Floriscan blooming-stage confusion")
fig.colorbar(im)
plt.tight_layout()
fig.savefig(EXPORT / "stage_confusion_matrix.png", dpi=140)
plt.show()
print("Copy this file to D:\\Projects\\Floriscan\\backend\\models\\floriscan_stage.pth")
print(best_path)
print("Flask cascade uses STAGE_SUPPORTED =", STAGE_SUPPORTED)
